# **Text classification using pretrained word embeddings**




In [31]:
from torchtext.vocab import GloVe, vocab

## **Step 1: Import Model**

In [32]:
glove_6B = GloVe(name="6B")

In [33]:
# build vocabulary
vocab = vocab(glove_6B.stoi,specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [34]:
vocab(["<unk>","Hello","hello"])

[0, 0, 13075]

## **Step 2: Tokenization**

In [50]:
from torchtext.data.utils import get_tokenizer
from torchtext.datasets import AG_NEWS
from torchtext.data.functional import to_map_style_dataset  
from torch.utils.data import random_split
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from tqdm import tqdm

In [36]:
tokenizer = get_tokenizer('basic_english')

Import Dataset

In [37]:
#initialize the train and test iterators
train_iter, test_iter = AG_NEWS()

# define train and test datasets
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)

Split train dataset into valid dataset also
 

In [38]:
num_train = int(len(train_dataset)*0.85) 

#randomly split train and valid datasets
split_train, split_valid = random_split(train_dataset,[num_train,(len(train_dataset)-num_train)])

In [39]:
# define class labels
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tec"}
num_class = len(set([label for (label,text) in train_iter]))
num_class

4

## **Step 3: Data Loader**

Define collate function

In [40]:
def text_pipeline(x):
    x = x.lower()
    return vocab(tokenizer(x))

def label_pipeline(x):
    return int(x)-1

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

In [42]:
def collate_func(batch):
    
    label_list, text_list, offsets = [],[],[0]
    
    for _label, _text in batch:
        label_list.append(label_pipeline(_label))
        process_text = torch.tensor(text_pipeline(_text),dtype=torch.int64)
        text_list.append(process_text)
        offsets.append(process_text.size(0))
    
    label_list = torch.tensor(label_list,dtype=torch.int64)
    offsets = torch.tensor(offsets).cumsum(dim=0)
    text_list = torch.cat(text_list)
    
    return label_list.to(device), text_list.to(device), offsets.to(device)

#### Define Data Loader

In [44]:
batch_size = 64

In [ ]:
train_datalaoder = DataLoader(
    split_train,shuffle=True, batch_size=batch_size, collate_fn=collate_func
)

valid_dataloader = DataLoader(
    split_valid, shuffle=True, batch_size=batch_size, collate_fn=collate_func
)

test_dataloader = DataLoader(
    test_dataset, shuffle=True, batch_size=batch_size, collate_fn=collate_func
)

## **Step 4: Model Class Design**

In [ ]:
class TextClassifierModel(nn.Module):
    def __init__(self, vocab_size, embedded_dim, num_class):
        super(TextClassifierModel,self).__init__()
        # define embedding layer from pre-define model
        self.embeddings = nn.Embedding.from_pretrained(glove_6B.vectors,freeze=True)
        
        #define fully connected layer
        self.fc = nn.Linear(in_features=embedded_dim, out_features=num_class)
        #define initial weights (random)
        self.init_weights()
    
    def init_weights(self):
        initrange = 0.5
        #define weights for embedding layer
        self.embeddings.weight.data.uniform_(-initrange,initrange)
        #define weights for fc layer
        self.fc.weight.data.uniform_(-initrange,initrange)
        #define initial bias zero in fc layer
        self.fc.bias.data.zero_()
    
    def forward(self, text,offset):
        mean = []
        #embedding layer out
        out_embedded = self.embeddings(text)
        #define mean embedding value for each sentence
        # this is similar to embedding-bag
        for i in range(1, len(offset)):
            embedding_values = out_embedded[offset[i-1],offset[i]]
            mean.append(embedding_values.mean(0))
            
        return self.fc(torch.stack(mean))

In [47]:
vocab_size = len(vocab)
emdedded_dim = 300

In [48]:
text_classifier = TextClassifierModel(vocab_size,emdedded_dim,num_class)

## **Step 5: Model Train**

Define training parameters

In [49]:
Lr = 5
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(text_classifier.parameters(),lr=Lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer,1,gamma=0.1)

Model Evaluation

In [51]:
def model_eval (dataloader,model):
    model.eval()
    
    total_acc, total_count =0.0
    
    for idx, (label,text,offset) in enumerate(dataloader):
        predicted_label = model(text,offset)
        
        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)
        
    return total_acc/total_count

In [52]:
def model_train (model, dataloader, criterion,optimizer,num_epochs = 1000):
    
    average_loss = []
    accuracy_epoch = []
    accuracy_old = 0
    
    for epoch in tqdm(range(num_epochs)):
    
        epoch_loss = 0
        model.train() 
        
        for idx, (label,text,offset) in enumerate(dataloader):
            
            optimizer.zero_grad()
            
            predicted_label = model(text,offset)
            loss = criterion(predicted_label,label)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1)
            
            optimizer.step()
            epoch_loss += loss.item()
        
        average_loss.append(epoch_loss/len(dataloader))
        acc_val = model_eval(split_valid,model)
        accuracy_epoch.append(acc_val)
        
        if acc_val > accuracy_old:
            accuracy_old = acc_val
            torch.save(model.state_dict(),"text-classifier.pth")
    return model, average_loss, accuracy_epoch
        